# Week 2 · Day 2 — LangChain: Tools, Chains, Memory & Your First Framework Agent

**Goal:** rebuild Day 1's raw-Python agent using LangChain, then go
further — chaining prompts, adding real memory, and giving the agent a
tool that touches an external data source.

**Backend used:** `langchain-google-genai` (Gemini), because it has a free
tier with no billing card required — the fastest path to a genuinely
working live run. Swapping to `langchain-anthropic`'s `ChatAnthropic` (or
any other LangChain chat model) is a one-line change in `lc_agent.py`;
every tool, chain, agent, memory, and structured-output piece below is
model-agnostic LangChain code.

**How to run this notebook (Google Colab recommended):**
1. Upload `lc_tools.py`, `lc_agent.py`, and `products.json` to the same
   Colab session folder as this notebook (drag them into the Colab file
   sidebar).
2. Get a free Gemini API key at https://aistudio.google.com/app/apikey
   (takes about a minute, no credit card).
3. Run the first code cell below, paste your key when prompted.
4. Runtime → Run all.

## Task 1 — LangChain Setup & Core Concepts

In [1]:
!pip install -q langchain langchain-classic langchain-core langchain-google-genai pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 11.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.1 which is incompatible.


In [2]:
import os
import getpass

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")

print("API key set:", bool(os.environ.get("GOOGLE_API_KEY")))

Enter your Gemini API key: ··········
API key set: True


### Mapping Day 1's raw-Python concepts onto LangChain

| Day 1 (raw Anthropic API) | LangChain equivalent |
|---|---|
| `anthropic.Anthropic()` client | `llm` — a `BaseChatModel`, e.g. `ChatGoogleGenerativeAI` |
| `TOOLS` list of hand-written JSON schemas + `TOOL_IMPLEMENTATIONS` dict | `@tool`-decorated Python functions — the docstring + type hints ARE the schema, generated automatically |
| `run_agent()` while-loop (reason → act → observe → repeat) | `AgentExecutor` — runs that identical loop for you |
| `messages` list (conversation memory, hand-appended every turn) | `RunnableWithMessageHistory` (modern) or `ConversationBufferMemory` (classic) |
| `working_memory` scratchpad + `log()` print statements | `AgentExecutor(verbose=True)` trace, or LangChain callbacks for more structured logging |
| Manually building a `tool_result` dict | Handled internally by `AgentExecutor` — you never touch a tool_result block directly |

The loop, the tool-result plumbing, and the memory bookkeeping we wrote by
hand on Day 1 are **exactly** what these LangChain primitives are doing
underneath — nothing here is a new concept, it's the same concepts with
the wiring hidden.

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)

# --- A basic LCEL prompt -> response pipeline --------------------------------
prompt = ChatPromptTemplate.from_messages(
    [("system", "You are a concise assistant."), ("human", "{question}")]
)
basic_chain = prompt | llm | StrOutputParser()

print(basic_chain.invoke({"question": "In one sentence, what is a ReAct agent?"}))

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


A ReAct agent is an AI system that solves complex tasks by continuously alternating between step-by-step reasoning ("thinking") and taking actions with external tools ("acting").


### What is the `|` pipe doing under the hood?

Every piece on either side of `|` — the prompt template, the chat model,
the output parser — implements the same `Runnable` interface (`.invoke()`,
`.stream()`, `.batch()`). LangChain overloads Python's bitwise-or operator
so that `a | b` returns a new `RunnableSequence` whose `.invoke(x)` simply
calls `b.invoke(a.invoke(x))`. So `prompt | llm | StrOutputParser()` is
function composition, not magic — it's equivalent to hand-writing
`StrOutputParser().invoke(llm.invoke(prompt.invoke({"question": ...})))`,
just with a much more readable syntax and consistent streaming/batching
behavior at every step for free.

## Task 2 — Define & Register Tools

Three tools, defined in `lc_tools.py` using LangChain's `@tool` decorator:

1. **`calculator`** — reused from Day 1.
2. **`get_weather`** — reused from Day 1 (fixed lookup table stub).
3. **`get_product_price`** — **new**: reads from a real external data
   source, a local JSON "database" (`products.json`), rather than the
   model's own knowledge.

**Why docstrings matter here specifically:** LangChain feeds each tool's
docstring — combined with its type-hinted function signature — directly
into the prompt the model sees, in exactly the slot Day 1's hand-written
`description` field occupied. The docstring isn't documentation for other
developers here; it *is* the interface the model uses to decide whether
and how to call the tool. A vague docstring is exactly as damaging as a
vague JSON-schema description was on Day 1.

In [5]:
from lc_tools import TOOLS, calculator, get_weather, get_product_price

for t in TOOLS:
    print(f"--- {t.name} ---")
    print("description:", t.description)
    print("args schema:", t.args)
    print()

--- calculator ---
description: Evaluate a basic arithmetic expression and return the numeric result.

Use this any time a calculation, comparison of numbers, or price
difference needs to be computed precisely instead of estimated.
Only supports +, -, *, /, %, ** and parentheses.
Input must be a valid arithmetic expression, e.g. '1499 - 549'.
Do NOT pass words — only digits, operators, and parentheses.
args schema: {'expression': {'title': 'Expression', 'type': 'string'}}

--- get_weather ---
description: Look up the current weather for a named city.

Returns the temperature in Celsius and a short condition string
(e.g. 'sunny', 'rainy'). This is a stub backed by a small fixed
lookup table (Lahore, Karachi, Islamabad, London, Dubai), not a
live weather feed. If the city is not in the table, this raises an
error rather than guessing — surface that error to the user instead
of inventing a number.
args schema: {'city': {'title': 'City', 'type': 'string'}}

--- get_product_price ---
descri

In [6]:
# Sanity check: call one tool directly (bypassing the agent) to confirm it
# reads real data from products.json, not made-up numbers.
print(get_product_price.invoke({"product_name": "UltraBook Pro"}))

{"product": "UltraBook Pro", "price_usd": 1499, "category": "laptop", "specs": "14-inch, 16GB RAM, 512GB SSD"}


## Task 3 — Build an Agent with `create_tool_calling_agent` / `AgentExecutor`

In [7]:
from lc_agent import build_agent_executor

agent_executor = build_agent_executor(llm, verbose=True)

trace_result = agent_executor.invoke(
    {"input": "Look up the price of the UltraBook Pro and the BudgetBook Lite, "
              "then tell me which one is cheaper and by how much."}
)
print("\nFINAL ANSWER:\n", trace_result["output"])



> Entering new AgentExecutor chain...


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Invoking: `get_product_price` with `{'product_name': 'UltraBook Pro'}`


{"product": "UltraBook Pro", "price_usd": 1499, "category": "laptop", "specs": "14-inch, 16GB RAM, 512GB SSD"}
Invoking: `get_product_price` with `{'product_name': 'BudgetBook Lite'}`


{"product": "BudgetBook Lite", "price_usd": 549, "category": "laptop", "specs": "14-inch, 8GB RAM, 256GB SSD"}

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Invoking: `calculator` with `{'expression': '1499 - 549'}`


950

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'The prices from the product catalog are:\n- **UltraBook Pro:** $1,499\n- **BudgetBook Lite:** $549\n\nThe **BudgetBook Lite** is cheaper by **$950**.', 'index': 0, 'extras': {'signature': 'EpwCCpkCARFNMg+kaPl6lsxa5gaVHRRva1ve0ToRWAcQWRvpA9ftVmGDcMWdLdsmtZT5/4mIvoATBmJohvVlDGECZOBDA2OgEY/dR1w7jnyVdv+rBXv6ZkMxvsNVlDE6DaZ5967m1awuY4GfjHj6XSxBOgH+hh8EdeMHhbvmK7zyPA+Tj3FA+iQgUC7v4kItINcIQFWrr107fygWdBUA/Jn/OgOcMoQEfVQ3XsRIEERjV3AGNeOCkiUB7pqmcn1iEROyr7fA3mqSDDIvTbqiiYjt1owbw8glGZpef/igECQjbHZTy6t111qtA/HI8bC+jWm6T/dRFGxQa1ywIKLyAiI7cnet6TnoGGs5JYibI5lp7NEW8wd59v9tes4kkBw='}}]

> Finished chain.

FINAL ANSWER:
 [{'type': 'text', 'text': 'The prices from the product catalog are:\n- **UltraBook Pro:** $1,499\n- **BudgetBook Lite:** $549\n\nThe **BudgetBook Lite** is cheaper by **$950**.', 'index': 0, 'extras': {'signature': 'EpwCCpkCARFNMg+kaPl6lsxa5gaVHRRva1ve0ToRWAcQWRvpA9ftVmGDcMWdLdsmtZT5/4mIvoATBmJohvVlDGECZOBDA2OgEY/dR1w7jnyVdv+rBXv6ZkMxvsNVlDE6DaZ5967m1awuY4Gf

### Annotating the trace above

With `verbose=True`, `AgentExecutor` prints its internal steps as it runs.
Reading the printed trace top to bottom, each block corresponds to:

- **REASON** — a green `> Entering new AgentExecutor chain...` block
  followed by the model's chosen action: `Invoking: get_product_price`
  with some input. This is the model deciding *what* to do next, same as
  the `[model reasoning/text]` line in Day 1's `run_agent()` log.
- **ACT** — LangChain calling the actual Python function behind
  `get_product_price` with the arguments the model chose. Equivalent to
  Day 1's `[ACT] calling tool '...' with input {...}` line.
- **OBSERVE** — the tool's return value being printed and fed back into
  the model's context automatically. Equivalent to Day 1's
  `[OBSERVE] result: ...` line.
- This repeats for the second product lookup (UltraBook Pro, then
  BudgetBook Lite) before the model has enough information to produce a
  final `Final Answer:` — exactly mirroring the two sequential tool calls
  in Day 1's weather-comparison task.

**What's similar to Day 1:** the underlying loop is identical —
reason → act → observe → repeat until no more tool calls are requested.

**What's now hidden:** the exact shape of the `tool_result` block, how the
model's tool-choice is parsed out of the raw API response, and how the
scratchpad of prior actions is re-serialized into the next prompt are all
handled internally by `AgentExecutor` / `create_tool_calling_agent`. On
Day 1 we could see and edit every one of those fields by hand; here they
exist, but only inside LangChain's internals unless you dig into its
source or attach custom callbacks.

## Task 4 — Add Memory

Wrapping the `AgentExecutor` in `RunnableWithMessageHistory` lets the agent
handle a 3-turn conversation where later turns depend on earlier context —
something a stateless single call cannot do.

Turn 2 only makes sense because the model can see Turn 1's tool result in
its `chat_history` (it knows "it" refers to the UltraBook Pro without
being told again). Turn 3 depends on both prior turns' context. This is
the direct LangChain equivalent of Day 1's `messages` list growing every
turn — except here `RunnableWithMessageHistory` manages that list for us,
keyed by `session_id`, instead of us appending to it by hand.

In [10]:
class SimpleMemoryAgent:
    """
    Sidesteps a LangChain/Gemini interop bug where AIMessage.content comes
    back as a list of blocks carrying internal 'thought signature' metadata,
    which breaks message coercion when replayed through MessagesPlaceholder.

    Fix: never build a list-of-BaseMessage chat history at all. Instead,
    keep a plain-text transcript and prepend it to the next turn's input
    string. No BaseMessage objects, no MessagesPlaceholder, so there is
    nothing left for the broken coercion path to touch.
    """
    def __init__(self, agent_executor):
        self.agent_executor = agent_executor
        self.turns = []  # list of (user_text, assistant_text) plain strings

    def invoke(self, inputs, config=None):
        history_text = "".join(
            f"User: {u}\nAssistant: {a}\n" for u, a in self.turns
        )
        full_input = (history_text + f"User: {inputs['input']}") if history_text else inputs["input"]
        result = self.agent_executor.invoke({"input": full_input})
        self.turns.append((inputs["input"], result["output"]))
        return result


agent_with_memory = SimpleMemoryAgent(agent_executor)

config = {"configurable": {"session_id": "demo"}}

r1 = agent_with_memory.invoke({"input": "Find the price of the UltraBook Pro."})
print("Turn 1:", r1["output"])

r2 = agent_with_memory.invoke({"input": "Now compare it to the BudgetBook Lite."})
print("Turn 2:", r2["output"])

r3 = agent_with_memory.invoke({"input": "Which one should I recommend to a budget-conscious client?"})
print("Turn 3:", r3["output"])



> Entering new AgentExecutor chain...


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Invoking: `get_product_price` with `{'product_name': 'UltraBook Pro'}`


{"product": "UltraBook Pro", "price_usd": 1499, "category": "laptop", "specs": "14-inch, 16GB RAM, 512GB SSD"}

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'The price of the UltraBook Pro is $1,499. It comes with a 14-inch display, 16GB RAM, and a 512GB SSD.', 'index': 0, 'extras': {'signature': 'EoICCv8BARFNMg/7/lDjGCEo+P4n6uUpKWRQ2nkUZuisYZlGprlmwuWm2LeOSbGVQpZSiC1Pc9ZF3lYV1xHTheHdvzlhxb9RE/KQRSP7/V4hLbza2VvuIkgUECbrfE74/0A2FlVwKPjsZc75SXSw88/54gMzWcO1AusAxYyPvoJL9xr8vO6VKGfi+91Riv/mZKXQL3ObvNag005C7XFsmtea/2ULc4+3iTfyUosqMRHntlo9eZ3V3pvmuKvEjMVHTCs0XmgFLQl93STNZ9/pHTWgqc8aZUmm+d64S7dKYOmKUjrp8Ak6HKTgHsBwXhxtgiSyWP3ZpC0NfvcjuRc5Y2fH0gis'}}]

> Finished chain.
Turn 1: [{'type': 'text', 'text': 'The price of the UltraBook Pro is $1,499. It comes with a 14-inch display, 16GB RAM, and a 512GB SSD.', 'index': 0, 'extras': {'signature': 'EoICCv8BARFNMg/7/lDjGCEo+P4n6uUpKWRQ2nkUZuisYZlGprlmwuWm2LeOSbGVQpZSiC1Pc9ZF3lYV1xHTheHdvzlhxb9RE/KQRSP7/V4hLbza2VvuIkgUECbrfE74/0A2FlVwKPjsZc75SXSw88/54gMzWcO1AusAxYyPvoJL9xr8vO6VKGfi+91Riv/mZKXQL3ObvNag005C7XFsmtea/2ULc4+3iTfyUosqMRHntlo9eZ3V3pvmuKvEjMVHTCs0XmgFLQl93STNZ9/pHTWgqc8a

...


Invoking: `calculator` with `{'expression': '1499 - 549'}`


950

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'The BudgetBook Lite costs **$549**, while the UltraBook Pro costs **$1,499**. \n\nHere is how they compare:\n\n* **Price Difference:** The UltraBook Pro is **$950** more expensive than the BudgetBook Lite ($1,499 vs. $549).\n* **RAM:** UltraBook Pro has 16GB RAM, compared to 8GB RAM on the BudgetBook Lite.\n* **Storage:** UltraBook Pro has a 512GB SSD, compared to a 256GB SSD on the BudgetBook Lite.\n* **Display:** Both feature a 14-inch display.', 'index': 0, 'extras': {'signature': 'Ep4ECpsEARFNMg/s5SgZ52Gu6qKwlr9HmZjQiawQc63knxFSGK5tSmBxivk1Wbb0VDY7lLW5uDY0kh1uaL2xH2zDpCDeyVmT1BfAzmlhinAql0GwDRo/3sEfPs1OSKtZGHJxPyG+7IkUt0/jQTq+yJZzSjoDD+2vkaNy0s3H/LavC7uDixZyALCd3R556c3mbXfA5IgdJvnnFJzORzpmLCjJONhi9vb557hGNeygj3aep/Qn5zy0yF+qpZQ79mGq1WaQgRp1VH3h5IlQ9m1mX1SrdCMbVUC4Myuil1QCGSAiIuNs4neG3+aP3SNL1kfgrV91Cq4LR7xoPzJ4oLKzTkLbeRovC2QugCFvXQrGRILChYZJsTxG3aqEMZvzveFTAJJaFykcL2vexjfMegg8QF1uOXv7xuKuN44aUQxWbmn/BOHOO0GCQRINzzRpWIfE0bADDILiMWRCuVtNPNrC9kNwL1Ghh8jwIbp

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'You should recommend the **BudgetBook Lite**. \n\nAt **$549**, it is **$950 cheaper** than the UltraBook Pro while still providing solid essential specs for everyday tasks, including a 14-inch display, 8GB of RAM, and a 256GB SSD. It offers great value for anyone prioritizing affordability.', 'index': 0, 'extras': {'signature': 'EtIJCs8JARFNMg/58wgHHr4Td2/MbpGsG9e/dvgJCUEFz7W444eaUy7zds7DsJVTtUBM0cEjG51cXm44FhsOnlTB7IRoHhPILq5Z58ZxLcE1BtZWz8GdKHyttTBb5qjnHaVhY8rL+QAhOT7aBpoUVt9nXt26rY5OUtH0Zivy9RQIehb8jn+oomHcx5Gi2gl4pEKGIXZmS5nnnpuFKWjDKx2POzOUrhO8jFq3poas/TPaevci2i48wEnjQ3HMRbVjccPjZLmKUTJik7ZvvOzpWHIUfESj/Bt1LN0iGFSu4yiH+3dCoVMms2nGzbKje5to1CxlGAgcFo3xzTwcKyEHPdmvwWwNe2jKLXheiHd15sN2yurZSk5crPLhNK27fPgdkZBvGXMOzzIhMxHZ+46HFvEGc0Q2h+iiJi+21lkmzJN+aP8wvTQxWAkLCmA6cwYr1THB1Ylt2MNgramGxTV4PN/mMSTo0Qy3FSkMhfqrQikHJn6dxW1CSoP4w2s+xy0UTuETE3IDFMCC6qM5VhbeFA0SSqY7CtTuUkUC4zNiIoyVh+sUknOySz4qsJ5ZtU6Ds1xT+ECLVbGsk9i4Naa6p+iQTjDf88BvW2aqLKsuJ11qqx5FM2QlGcr/O55tMv3JUV

## Task 5 — Structured Output & Error Handling

### Structured output

In [12]:
from lc_agent import build_structured_formatter, Recommendation

structured_formatter = build_structured_formatter(llm)
structured_result = structured_formatter.invoke({"agent_answer": r3["output"]})

print(type(structured_result))
print(structured_result)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


<class 'lc_agent.Recommendation'>
recommended_product='BudgetBook Lite' price_usd=549.0 price_difference_usd=950.0 reasoning='It provides solid essential specs for everyday tasks while being $950 cheaper than the UltraBook Pro, making it a great value for anyone prioritizing affordability.'


`with_structured_output(Recommendation)` forces the model's response into
the `Recommendation` Pydantic model (`recommended_product`, `price_usd`,
`price_difference_usd`, `reasoning`) — LangChain handles generating the
schema instructions and parsing/validating the model's output back into a
real Python object, raising a validation error if the model's output
doesn't fit the schema.

### Error handling: a tool that sometimes fails

In [14]:
# get_product_price raises KeyError for unknown products — trigger that
# real failure through the full agent loop and observe how it's handled.
error_result = agent_executor.invoke(
    {"input": "What's the price of the 'Quantum Laptop 9000'?"}
)
print("\nFINAL ANSWER AFTER TOOL ERROR:\n", error_result["output"])



> Entering new AgentExecutor chain...


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


GoogleRateLimitError: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 33.928721997s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', '

**How the agent recovers:** `get_product_price` raises a plain `KeyError`
with a descriptive message (unchanged from Day 1's philosophy — never fail
silently). `AgentExecutor` catches that exception internally, converts it
into an error string, and feeds it back to the model as the tool's
observation instead of letting the whole chain crash. The model then sees
a clear error message (including the list of known products) and responds
by explaining it couldn't find that product, rather than inventing a
price. The only configuration needed was `handle_parsing_errors=True` on
`AgentExecutor` (in `lc_agent.py`) to also cover the case where the
model's own tool-call formatting is malformed, not just tool-execution
errors.

### LangChain vs. Day 1's raw agent — what changed

LangChain made tool registration effectively free (a docstring instead of
hand-written JSON schema), collapsed the entire reason/act/observe loop
into `AgentExecutor`, and made multi-turn memory a two-line wrapper
(`RunnableWithMessageHistory`) instead of manually managing a growing
`messages` list. The abstraction leakage shows up in the trace: on Day 1
every tool_result block, every raw model response, and every loop
iteration was visible and editable; here, the same information exists but
is buried inside `AgentExecutor`'s internals unless `verbose=True` or a
custom callback is attached, and the exact retry/error-formatting logic
that decides *how* a tool exception gets turned into text for the model is
no longer something we wrote or fully control.